# Graph data download: Open Payments + Part D by drug
Run each cell in order. Total time is about 30-60 minutes because the source files are large. Each file is downloaded, shrunk with DuckDB, and deleted, so Colab's disk never fills up. The last cell downloads one small zip to your computer.

In [ ]:
# Step 1: install DuckDB
!pip -q install duckdb

In [ ]:
# Step 2: settings and helper functions
import os
import shutil
from pathlib import Path

import duckdb
import requests

YEARS = [2022, 2023, 2024]
TMP = Path("/content/tmp")
OUT = Path("/content/graph_data")
TMP.mkdir(parents=True, exist_ok=True)
(OUT / "openpayments").mkdir(parents=True, exist_ok=True)
(OUT / "partd_drug").mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"
    )
}

con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")


def download(url, dest):
    print(f"  downloading {url.rsplit('/', 1)[-1]}")
    with requests.get(url, headers=HEADERS, stream=True, timeout=1800) as r:
        r.raise_for_status()
        done = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 << 20):
                f.write(chunk)
                done += len(chunk)
                print(f"\r  {done / 1e9:.2f} GB", end="")
    print()


def find_col(columns, needle):
    for c in columns:
        if needle in c:
            return c
    raise KeyError(f"No column containing {needle!r}. Columns: {columns}")


def csv_columns(path):
    rel = f"read_csv('{path}', header = true, all_varchar = true, normalize_names = true, ignore_errors = true)"
    cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {rel}").fetchall()]
    return rel, cols


print("Settings loaded")

In [ ]:
# Step 3: download links
# Open Payments links (found on openpaymentsdata.cms.gov, published 2026-06-30)
op_links = {
    2022: "https://download.cms.gov/openpayments/PGYR2022_P06302026_06032026/OP_DTL_GNRL_PGYR2022_P06302026_06032026.csv",
    2023: "https://download.cms.gov/openpayments/PGYR2023_P06302026_06032026/OP_DTL_GNRL_PGYR2023_P06302026_06032026.csv",
    2024: "https://download.cms.gov/openpayments/PGYR2024_P06302026_06032026/OP_DTL_GNRL_PGYR2024_P06302026_06032026.csv",
}

# Check that Colab can reach the Open Payments download server
check = requests.head(op_links[2024], headers=HEADERS, timeout=60, allow_redirects=True)
size_gb = int(check.headers.get("content-length", 0)) / 1e9
print(f"Open Payments server: HTTP {check.status_code}, 2024 file is {size_gb:.1f} GB")
if check.status_code != 200:
    print("BLOCKED: Runtime > Disconnect and delete runtime, then run all cells again.")

# Part D by provider and drug links from the CMS catalog
response = requests.get("https://data.cms.gov/data.json", headers=HEADERS, timeout=300)
if not response.ok or not response.text.lstrip().startswith("{"):
    print(f"CMS catalog blocked (HTTP {response.status_code}): {response.text[:150]}")
    print("BLOCKED: Runtime > Disconnect and delete runtime, then run all cells again.")
else:
    catalog = response.json()["dataset"]
    drug_links = {}
    for ds in catalog:
        series = next((s["title"] for s in ds.get("inSeries") or [] if s.get("title")),
                      ds.get("title", "").split(" : ")[0])
        if series.strip() != "Medicare Part D Prescribers - by Provider and Drug":
            continue
        end = (ds.get("temporal") or [{}])[0].get("endDate", "")
        for dist in ds.get("distribution", []):
            url = dist.get("downloadURL", "")
            if dist.get("format") == "CSV" and url.lower().endswith(".csv") and end[:4].isdigit():
                drug_links[int(end[:4])] = url
    print("Part D by drug years:", sorted(drug_links))
    missing = [y for y in YEARS if y not in drug_links]
    print("Missing years:", missing or "none")

In [ ]:
# Step 4: Open Payments -> one row per doctor, company, payment type and product
def shrink_open_payments(csv_path, year, out_path):
    rel, cols = csv_columns(csv_path)
    npi = find_col(cols, "covered_recipient_npi")
    company = find_col(cols, "making_payment_name")
    nature = find_col(cols, "nature_of_payment")
    amount = find_col(cols, "total_amount_of_payment")
    product = find_col(cols, "name_of_drug_or_biological_or_device_or_medical_supply_1")
    ptype = find_col(cols, "indicate_drug_or_biological_or_device_or_medical_supply_1")
    con.execute(f"""
        COPY (
            SELECT TRIM({npi}) AS npi,
                   {year} AS year,
                   TRIM({company}) AS company,
                   TRIM({nature}) AS nature,
                   UPPER(TRIM({product})) AS product,
                   TRIM({ptype}) AS product_type,
                   COUNT(*) AS n_payments,
                   SUM(TRY_CAST({amount} AS DOUBLE)) AS usd
            FROM {rel}
            WHERE REGEXP_FULL_MATCH(TRIM({npi}), '[0-9]{{10}}')
            GROUP BY ALL
        ) TO '{out_path}' (FORMAT parquet, COMPRESSION zstd)
    """)
    rows = con.execute(f"SELECT COUNT(*), ROUND(SUM(usd) / 1e6, 1) FROM '{out_path}'").fetchone()
    print(f"  saved {out_path.name}: {rows[0]:,} rows, ${rows[1]:,}M total")


for year in YEARS:
    out_path = OUT / "openpayments" / f"openpayments_{year}.parquet"
    if out_path.exists():
        print(f"{year}: already done")
        continue
    print(f"Open Payments {year}")
    csv_path = TMP / f"op_{year}.csv"
    download(op_links[year], csv_path)
    shrink_open_payments(csv_path, year, out_path)
    os.remove(csv_path)

In [ ]:
# Step 5: Part D by provider and drug -> brand-name drugs only
def shrink_partd_drug(csv_path, year, out_path):
    rel, cols = csv_columns(csv_path)
    con.execute(f"""
        COPY (
            SELECT TRIM(prscrbr_npi) AS npi,
                   {year} AS year,
                   UPPER(TRIM(brnd_name)) AS brand,
                   UPPER(TRIM(gnrc_name)) AS generic,
                   TRY_CAST(NULLIF(TRIM(tot_clms), '') AS DOUBLE) AS claims,
                   TRY_CAST(NULLIF(TRIM(tot_drug_cst), '') AS DOUBLE) AS drug_cost
            FROM {rel}
            WHERE UPPER(TRIM(brnd_name)) <> UPPER(TRIM(gnrc_name))
        ) TO '{out_path}' (FORMAT parquet, COMPRESSION zstd)
    """)
    rows = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT brand) FROM '{out_path}'").fetchone()
    print(f"  saved {out_path.name}: {rows[0]:,} rows, {rows[1]:,} brand drugs")


for year in YEARS:
    out_path = OUT / "partd_drug" / f"partd_drug_{year}.parquet"
    if out_path.exists():
        print(f"{year}: already done")
        continue
    print(f"Part D by drug {year}")
    csv_path = TMP / f"drug_{year}.csv"
    download(drug_links[year], csv_path)
    shrink_partd_drug(csv_path, year, out_path)
    os.remove(csv_path)

In [ ]:
# Step 6: zip and download to your computer
from google.colab import files

shutil.make_archive("/content/graph_data", "zip", OUT)
size_mb = Path("/content/graph_data.zip").stat().st_size / 1e6
print(f"Zip created ({size_mb:,.0f} MB), download starting...")
files.download("/content/graph_data.zip")